## SRP336229

**paper:** [PMID: 35676317](https://pmc.ncbi.nlm.nih.gov/articles/PMC9177602/) - Seasonal and sex-dependent gene expression in emu (Dromaius novaehollandiae) fat tissues, 2022

**date, curator:** 2026-03-09, Sara Carsanaro

**notes**
* in methods they reference [this paper](https://www.jscimedcentral.com/public/assets/articles/vascularmedicine-7-1111.pdf) for library prep, TruSeq RNA sample prep kit v2
* for anatomical entity:
    * likely should be retroperitoneal fat pad for back fat pad
    * likely should be abdominal fat pad for just abdominal fat pad i believe
    * parent term is abdominal fat pad - i guess this is the most appropriate term here

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,fat - back and abdominal fat pads,UBERON:0003427,abdominal fat pad,other


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP336229"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 24/24 [00:27<00:00,  1.13s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,,,,,,fat - back and abdominal fat pads,adult,,,,M,,,8790,,,,,,K34-12022,SAMN21362029,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K34-12022,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819947,52CDD788268780E115A4294AC9C248B1
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,,,,,,fat - back and abdominal fat pads,adult,,,,M,,,8790,,,,,,K33-12016,SAMN21362028,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K33-12016,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819948,DEEF195FE838A26C86FDA3BF1B183226
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,,,,,,fat - back and abdominal fat pads,adult,,,,F,,,8790,,,,,,K32-12057,SAMN21362027,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K32-12057,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819949,D776A95B49D7C692555311DE3F5AF3CE
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,,,,,,fat - back and abdominal fat pads,adult,,,,F,,,8790,,,,,,K31-12051,SAMN21362026,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K31-12051,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819950,26BF53EFC307C7C6700E770BDFAA8461
4,SRX12112058,SRP336229,Illumina MiSeq,SRS10092314,,,,,,fat - back and abdominal fat pads,adult,,,,F,,,8790,,,,,,K30-12045,SAMN21362025,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 qua

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['fat - back and abdominal fat pads']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0003427'
library.loc[:,'anatName'] = 'abdominal fat pad'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'other'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,,,,fat - back and abdominal fat pads,adult,other,not documented,,M,,,8790,,,,,,K34-12022,SAMN21362029,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K34-12022,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819947,52CDD788268780E115A4294AC9C248B1
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,,,,fat - back and abdominal fat pads,adult,other,not documented,,M,,,8790,,,,,,K33-12016,SAMN21362028,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K33-12016,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819948,DEEF195FE838A26C86FDA3BF1B183226
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,,,,fat - back and abdominal fat pads,adult,other,not documented,,F,,,8790,,,,,,K32-12057,SAMN21362027,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K32-12057,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819949,D776A95B49D7C692555311DE3F5AF3CE
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,,,,fat - back and abdominal fat pads,adult,other,not documented,,F,,,8790,,,,,,K31-12051,SAMN21362026,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K31-12051,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819950,26BF53EFC307C7C6700E770BDFAA8461
4,SRX12112058,SRP336229,Illumina MiSeq,SRS10092314,UBERON:0003427,abdominal fat pad,,,,fat - back and abdominal fat pads,adult,other,not documented,,F,,,8790,,,,,,K30-12045,SAMN21362025,,,,,,,,,,,09/03/2026,"Each fat sa

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,,,,,,K34-12022,SAMN21362029,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K34-12022,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819947,52CDD788268780E115A4294AC9C248B1
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,,,,,,K33-12016,SAMN21362028,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K33-12016,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819948,DEEF195FE838A26C86FDA3BF1B183226
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,,,,,,K32-12057,SAMN21362027,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K32-12057,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819949,D776A95B49D7C692555311DE3F5AF3CE
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,,,,,,K31-12051,SAMN21362026,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K31-12051,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819950,26BF53EFC307C7C6700E770BDFAA8461
4,SRX121120

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K34-12022,SAMN21362029,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K34-12022,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819947,52CDD788268780E115A4294AC9C248B1
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K33-12016,SAMN21362028,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K33-12016,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819948,DEEF195FE838A26C86FDA3BF1B183226
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K32-12057,SAMN21362027,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K32-12057,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819949,D776A95B49D7C692555311DE3F5AF3CE
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K31-12051,SAMN21362026,,,,,,,,,,,09/03/2026,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-proces

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K34-12022,SAMN21362029,,,,,,,,,,SAC,2026-03-10,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K34-12022,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819947,52CDD788268780E115A4294AC9C248B1
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K33-12016,SAMN21362028,,,,,,,,,,SAC,2026-03-10,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K33-12016,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819948,DEEF195FE838A26C86FDA3BF1B183226
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K32-12057,SAMN21362027,,,,,,,,,,SAC,2026-03-10,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequence pre-processing software Trimmomatic (v0.30), was used to obtain clean paired-end and single-end Mi-Seq data.",,K32-12057,,,,adult,,TRANSCRIPTOMIC,PCR,SRR15819949,D776A95B49D7C692555311DE3F5AF3CE
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K31-12051,SAMN21362026,,,,,,,,,,SAC,2026-03-10,"Each fat sample (a pool of six biopsies from different spots of the back and abdominal fat pads) was homogenized and prepared for RNA-sequencing. Illumina Mi-Seq system was used to sequence the emu fat transcriptome library. Raw reads were filtered with Q20 quality trimming to remove low quality reads with average quality score <20 and trimming of low-quality bases from the end of reads. Sequenc

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 35676317'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K34-12022,SAMN21362029,,,,,,,PMID: 35676317,,,SAC,2026-03-10
1,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K33-12016,SAMN21362028,,,,,,,PMID: 35676317,,,SAC,2026-03-10
2,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K32-12057,SAMN21362027,,,,,,,PMID: 35676317,,,SAC,2026-03-10
3,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K31-12051,SAMN21362026,,,,,,,PMID: 35676317,,,SAC,2026-03-10
4,SRX12112058,SRP336229,Illumina MiSeq,SRS10092314,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K30-12045,SAMN21362025,,,,,,,PMID: 35676317,,,SAC,2026-03-10
5,SRX12112057,SRP336229,Illumina MiSeq,SRS10092313,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K29-12039,SAMN21362024,,,,,,,PMID: 35676317,,,SAC,2026-03-10
6,SRX12112056,SRP336229,Illumina MiSeq,SRS10092312,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K28-12033,SAMN21362023,,,,,,,PMID: 35676317,,,SAC,2026-03-10
7,SRX12112055,SRP336229,Illumina MiSeq,SRS10092311,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K48-12059,SAMN21362043,,,,,,,PMID: 35676317,,,SAC,2026-03-10
8,SRX12112054,SRP336229,Illumina MiSeq,SRS10092310,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K47-12053,SAMN21362042,,,,,,,PMID: 35676317,,,SAC,2026-03-10
9,SRX12112053,SRP336229,Illumina MiSeq,SRS10092309,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K46_12047,SAMN21362041,,,,,,,PMID: 35676317,,,SAC,2026-03-10


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP336229,Seasonal and sex- dependent gene expression in emu (Dromaius novaehollandiae) fat tissues,"Emus (Dromaius novaehollandiae) are birds farmed for fat production. Oil rendered from their fat is valued for its anti-inflammatory and antioxidant properties for uses in therapeutics and cosmetics. We analyzed the seasonal and sex-dependent differentially expressed (DE) genes involved in fat metabolism. Samples were taken from back and abdominal fat tissues of the same four male and four female emus in April, June, and November for RNA-sequencing.",SRA,,,,,,,PRJNA761725,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

24

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP336229,Seasonal and sex- dependent gene expression in emu (Dromaius novaehollandiae) fat tissues,"Emus (Dromaius novaehollandiae) are birds farmed for fat production. Oil rendered from their fat is valued for its anti-inflammatory and antioxidant properties for uses in therapeutics and cosmetics. We analyzed the seasonal and sex-dependent differentially expressed (DE) genes involved in fat metabolism. Samples were taken from back and abdominal fat tissues of the same four male and four female emus in April, June, and November for RNA-sequencing.",SRA,total,Bgee 1K,24,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA761725,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '35676317'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9177602/'
experiment.loc[:,'DOI'] = '10.1038/s41598-022-13681-5'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP336229,Seasonal and sex- dependent gene expression in emu (Dromaius novaehollandiae) fat tissues,"Emus (Dromaius novaehollandiae) are birds farmed for fat production. Oil rendered from their fat is valued for its anti-inflammatory and antioxidant properties for uses in therapeutics and cosmetics. We analyzed the seasonal and sex-dependent differentially expressed (DE) genes involved in fat metabolism. Samples were taken from back and abdominal fat tissues of the same four male and four female emus in April, June, and November for RNA-sequencing.",SRA,total,Bgee 1K,24,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA761725,35676317,https://pmc.ncbi.nlm.nih.gov/articles/PMC9177602/,10.1038/s41598-022-13681-5,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
55792,SRX9686047,SRP297984,Illumina HiSeq 4000,SRS7883668,UBERON:6003005,insect adult tagma,UBERON:0000066,fully formed stage,,head and thorax,maximum of 22 months reproducing,other,not documented,other,F,,,105785,TruSeq RNA Library Prep Kit,full_length,polyA,,,"Generic sample from Cryptotermes secundus, que...",SAMN17081623,,,,,,,"PMID:33753888,… the data represent independent...",neotenic queen,,ANN,2026-03-10
55793,SRX9686041,SRP297984,Illumina HiSeq 4000,SRS7883662,UBERON:6003005,insect adult tagma,UBERON:0000066,fully formed stage,,head and thorax,maximum of 22 months reproducing,other,not documented,other,F,,,105785,TruSeq RNA Library Prep Kit,full_length,polyA,,,"Generic sample from Cryptotermes secundus, que...",SAMN17081622,,,,,,,"PMID:33753888,… the data represent independent...",neotenic queen,,ANN,2026-03-10
55794,SRX12112062,SRP336229,Illumina MiSeq,SRS10092318,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K34-12022,SAMN21362029,,,,,,,PMID: 35676317,,,SAC,2026-03-10
55795,SRX12112061,SRP336229,Illumina MiSeq,SRS10092317,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,M,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K33-12016,SAMN21362028,,,,,,,PMID: 35676317,,,SAC,2026-03-10
55796,SRX12112060,SRP336229,Illumina MiSeq,SRS10092316,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K32-12057,SAMN21362027,,,,,,,PMID: 35676317,,,SAC,2026-03-10
55797,SRX12112059,SRP336229,Illumina MiSeq,SRS10092315,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K31-12051,SAMN21362026,,,,,,,PMID: 35676317,,,SAC,2026-03-10
55798,SRX12112058,SRP336229,Illumina MiSeq,SRS10092314,UBERON:0003427,abdominal fat pad,UBERON:0000113,post-juvenile adult stage,,fat - back and abdominal fat pads,adult,other,not documented,perfect match,F,,,8790,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,K30-12045,SAMN21362025,,,,,,,PMID: 35676317,,,SAC,2026-03-10


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1083,SRP301570,Disentangling the aging network of a termite q...,We studied aging in queens of the drywood term...,SRA,total,So-Long Project; Bgee 1K,12,TruSeq RNA Library Prep Kit,full_length,,PRJNA691762,33975542,https://pmc.ncbi.nlm.nih.gov/articles/PMC8114706/,10.1186/s12864-021-07649-4,,Samples ‘brain and thorax (including legs) (SR...
1084,SRP297984,"Transcriptomic analyses of the termite, Crypto...",This study identified a single gene network un...,SRA,partial,So-Long Project; Bgee 1K,6,TruSeq RNA Library Prep Kit,full_length,,PRJNA685294,33753888,https://pmc.ncbi.nlm.nih.gov/articles/PMC7985136/,10.1038/s42003-021-01892-x,,termites ‘head and thorax’ samples
1085,SRP336229,Seasonal and sex- dependent gene expression in...,Emus (Dromaius novaehollandiae) are birds farm...,SRA,total,Bgee 1K,24,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA761725,35676317,https://pmc.ncbi.nlm.nih.gov/articles/PMC9177602/,10.1038/s41598-022-13681-5,,


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 0ba08e4] adding annotated bulk experiment SRP336229
 2 files changed, 25 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.36 KiB | 2.36 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   2eea986..0ba08e4  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push